In [3]:
import os
import cv2
import numpy as np

def ensure_dir(directory):
    """Создает папку, если ее не существует."""
    if not os.path.exists(directory):
        os.makedirs(directory)

def binarize_and_clean(window, method='otsu'):
    """
    Бинаризует фрагмент изображения и применяет морфологическую очистку.
    Возвращает бинарное изображение window_bin.
    
    Параметры:
      window - фрагмент в градациях серого
      method - 'otsu' (глобальный порог Оцу) или 'adaptive' (адаптивный порог)

    Дополнительно можно настроить морф. операции и ядро по вкусу.
    """
    # Бинаризация
    if method == 'adaptive':
        # Адаптивный порог
        window_bin = cv2.adaptiveThreshold(window, 255,
                                           cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                           cv2.THRESH_BINARY, 11, 2)
    else:
        # Otsu
        _, window_bin = cv2.threshold(window, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Морфологическая очистка (пример: открытие, потом закрытие)
    kernel = np.ones((3,3), np.uint8)
    opened = cv2.morphologyEx(window_bin, cv2.MORPH_OPEN, kernel)
    cleaned = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return cleaned

def sliding_window_morph_diff(f_img, g_img, window_size=32, step=16,
                              diff_threshold=0.3, bin_method='otsu'):
    """
    Вариант 2 (скользящее окно + чисто морфологическое выделение отличий).
    
    Для каждого окна:
      1) Бинаризация фрагментов f_win и g_win + морф. очистка
      2) Морфологическая (бинарная) симметрическая разность
      3) Если доля белых пикселей в разности > diff_threshold => отличие

    Параметры:
      f_img, g_img       - изображения одинакового размера (градации серого)
      window_size        - размер окна (в пикселях)
      step               - шаг скольжения окна (может быть меньше window_size)
      diff_threshold     - порог доли отличающихся пикселей (0..1)
      bin_method         - 'otsu' или 'adaptive' (метод пороговой бинаризации)

    Возвращает:
      - result: цветная копия f с пометками красным
      - diff_map: карта (0/255), где окна с превышающей порог разницей помечаются белым
    """
    h, w = f_img.shape[:2]
    # Для визуализации сделаем BGR-копию f
    result = cv2.cvtColor(f_img, cv2.COLOR_GRAY2BGR)
    diff_map = np.zeros_like(f_img, dtype=np.uint8)
    
    for y in range(0, h - window_size + 1, step):
        for x in range(0, w - window_size + 1, step):
            # Фрагменты
            f_win = f_img[y:y+window_size, x:x+window_size]
            g_win = g_img[y:y+window_size, x:x+window_size]
            
            # Бинаризация и морфологическая очистка
            f_bin = binarize_and_clean(f_win, method=bin_method)
            g_bin = binarize_and_clean(g_win, method=bin_method)
            
            # Симметрическая разность
            # diff = (f_bin AND NOT g_bin) OR (g_bin AND NOT f_bin)
            diff_fg = cv2.bitwise_and(f_bin, cv2.bitwise_not(g_bin))
            diff_gf = cv2.bitwise_and(g_bin, cv2.bitwise_not(f_bin))
            sym_diff = cv2.bitwise_or(diff_fg, diff_gf)
            
            # Доля белых пикселей
            diff_norm = np.sum(sym_diff > 0) / float(window_size * window_size)
            
            if diff_norm > diff_threshold:
                # Отмечаем центр окна на result
                center_x = x + window_size // 2
                center_y = y + window_size // 2
                cv2.circle(result, (center_x, center_y), radius=3, color=(0,0,255), thickness=-1)
                # Заливаем окно белым на карте diff_map
                diff_map[y:y+window_size, x:x+window_size] = 255
    
    return result, diff_map

def main():
    # Пути к изображениям f и g
    f_path = os.path.join("source_images", "f.png")
    g_path = os.path.join("source_images", "g.png")
    
    if not os.path.exists(f_path) or not os.path.exists(g_path):
        print("Не найдены f.png / g.png в папке source_images.")
        return
    
    f_img = cv2.imread(f_path, cv2.IMREAD_GRAYSCALE)
    g_img = cv2.imread(g_path, cv2.IMREAD_GRAYSCALE)
    
    if f_img is None or g_img is None:
        print("Ошибка загрузки изображений.")
        return
    
    # Проверяем, что размеры совпадают
    if f_img.shape != g_img.shape:
        print("Размеры f и g не совпадают.")
        return
    
    # Параметры скользящего окна
    window_size = 32
    step = 16
    
    # Пороговая доля отличий
    diff_threshold = 0.3
    
    # Вычисляем карту отличий
    result_img, diff_map = sliding_window_morph_diff(
        f_img, g_img,
        window_size=window_size,
        step=step,
        diff_threshold=diff_threshold,
        bin_method='otsu'  # или 'adaptive'
    )
    
    # Сохраним результаты
    out_dir = "results/task8_variant2_morph"
    ensure_dir(out_dir)
    
    cv2.imwrite(os.path.join(out_dir, "result.png"), result_img)
    cv2.imwrite(os.path.join(out_dir, "diff_map.png"), diff_map)
    
    print("Обработка завершена. Результаты сохранены в:", out_dir)

if __name__ == "__main__":
    main()


Обработка завершена. Результаты сохранены в: results/task8_variant2_morph
